# Blockchain Lab 1 – Akcasu

## Overview

In this lab we build a minimal blockchain **from scratch** using only Python's standard library.
By the end you will understand:

| Part | Topic |
|------|-------|
| 1 | SHA-256 cryptographic hashing |
| 2 | Block structure and chain linkage |
| 3 | Chain validation and tamper detection |
| 4 | Proof of Work (PoW) mining |
| 5 | Forking and longest-chain selection |
| Extension | Logical transaction validity rules |

> **How to use this notebook:** Run each cell top-to-bottom in order.
> Every cell builds on the functions and variables defined above it.

---
## Part 1 – SHA-256 Hashing

### Background

A **cryptographic hash function** maps an input of any length to a fixed-length output called a *digest* or *hash*.
SHA-256 (Secure Hash Algorithm 256-bit) is the function used by Bitcoin and many other blockchains.

Key properties to remember:

- **Deterministic** – the same input always produces the same hash.
- **One-way (pre-image resistant)** – you cannot reverse-engineer the input from the hash.
- **Avalanche effect** – changing even a single character in the input produces a completely different hash.
- **Collision resistant** – it is computationally infeasible to find two different inputs with the same hash.

### Task
1. Hash three sample transactions and observe the 64-character hex output.
2. Change one character in a transaction and confirm the hash changes completely.

In [ ]:
# ── Part 1: Imports and SHA-256 Hash Function ────────────────────────────────
import hashlib
import json
from copy import deepcopy

def sha256_text(text: str) -> str:
    """Return the SHA-256 hex digest of a UTF-8 encoded string."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

transactions = [
    "Alice pays Bob 10",
    "Bob pays Carol 4",
    "Carol pays David 2",
]

print("Transaction → SHA-256 Hash")
print("-" * 60)
for tx in transactions:
    print(f"Input : {tx}")
    print(f"Hash  : {sha256_text(tx)}")
    print()

In [ ]:
# ── Part 1: Avalanche Effect Demo ────────────────────────────────────────────
# Changing only the amount from 10 → 11 produces a completely different hash.
original = "Alice pays Bob 10"
changed  = "Alice pays Bob 11"

print("Avalanche effect – one character changed:")
print(f"  Original (10) : {sha256_text(original)}")
print(f"  Changed  (11) : {sha256_text(changed)}")

---
## Part 2 – Block Structure and Chain Linkage

### Background

A **block** is a data container.  Every block in our chain stores:

```
{
  index         : integer position in the chain (0, 1, 2, …)
  timestamp     : ISO-8601 creation time
  transactions  : list of transaction strings
  previous_hash : hash of the block directly before this one
  nonce         : number adjusted during mining (default 0)
  hash          : SHA-256 of all fields above
}
```

The **genesis block** (index 0) has no predecessor, so its `previous_hash` is a string of 64 zeros.

Blocks are linked together because each block's `previous_hash` references the hash of the block before it.
This forms an **immutable chain**: altering any block changes its hash, which breaks the link to the next block.

### Task
Build a 3-block chain (genesis + block 1 + block 2) and display a summary of each block.

In [ ]:
# ── Part 2: Block Structure and Chain Linkage ────────────────────────────────
def calculate_block_hash(block: dict) -> str:
    """Compute SHA-256 over all block fields except the stored hash."""
    data    = {k: v for k, v in block.items() if k != "hash"}
    encoded = json.dumps(data, sort_keys=True, separators=(",", ":"))
    return sha256_text(encoded)

def make_block(
    index: int,
    transactions: list,
    previous_hash: str,
    timestamp: str,
    nonce: int = 0,
) -> dict:
    """Create and return a new block dictionary with its hash computed."""
    block = {
        "index":         index,
        "timestamp":     timestamp,
        "transactions":  transactions,
        "previous_hash": previous_hash,
        "nonce":         nonce,
    }
    block["hash"] = calculate_block_hash(block)
    return block

def show_chain(blocks: list) -> None:
    """Print a summary row for each block in the chain."""
    for block in blocks:
        print(
            f"  Block {block['index']}"
            f" | Hash: {block['hash'][:12]}…"
            f" | Prev: {block['previous_hash'][:12]}…"
            f" | Nonce: {block['nonce']}"
        )

# Build the initial 3-block chain
genesis = make_block(0, ["Course blockchain begins"], "0" * 64, "2026-10-01T09:00:00Z")
block_1 = make_block(1, ["Alice pays Bob 10", "Bob pays Carol 4"],  genesis["hash"], "2026-10-01T09:01:00Z")
block_2 = make_block(2, ["Carol pays David 2"], block_1["hash"], "2026-10-01T09:02:00Z")

chain = [genesis, block_1, block_2]

print("Initial 3-block chain:")
show_chain(chain)

---
## Part 3 – Chain Validation and Tamper Detection

### Background

Chain validation checks two guarantees for **every** block:

1. **Integrity check** – recompute the block's hash from its contents and compare it with the stored `hash` field.  
   → Detects any modification to the block's own data.

2. **Linkage check** – confirm that `previous_hash` equals the `hash` of the preceding block.  
   → Detects breaks in the chain caused by inserting or removing blocks.

### Why can't an attacker just fix the hash?

If an attacker modifies `block_1`, they must also:
- Recompute `block_1.hash`
- Update `block_2.previous_hash` to match the new `block_1.hash`
- Recompute `block_2.hash`
- … and so on for every subsequent block, **and** redo all the Proof of Work.

In a real distributed network this is computationally infeasible unless the attacker controls >50% of the network's hashing power.

### Task
1. Validate the original chain.
2. Tamper with a transaction and re-validate.
3. Patch only the tampered block's hash and show the linkage check still catches it.

In [ ]:
# ── Part 3: Chain Validation and Tamper Detection ────────────────────────────
def validate_chain(blocks: list) -> tuple:
    """Return (True, message) if the chain is intact, (False, reason) otherwise."""
    for position, block in enumerate(blocks):
        recalculated = calculate_block_hash(block)
        if block["hash"] != recalculated:
            return False, f"Block {position} has changed contents"
        if position > 0:
            if block["previous_hash"] != blocks[position - 1]["hash"]:
                return False, f"Block {position} has a broken previous-hash link"
    return True, "Chain is valid"

# 1. Original chain should be valid
print("1. Original chain          :", validate_chain(chain))

# 2. Tamper: change 10 → 100 in block 1
tampered_chain = deepcopy(chain)
tampered_chain[1]["transactions"][0] = "Alice pays Bob 100"
print("2. After tampering data    :", validate_chain(tampered_chain))

# 3. Attacker patches only block 1's hash – linkage check exposes the break
tampered_chain[1]["hash"] = calculate_block_hash(tampered_chain[1])
print("3. After re-hashing blk 1  :", validate_chain(tampered_chain))

---
## Part 4 – Proof of Work Mining

### Background

**Proof of Work (PoW)** is a consensus mechanism that makes adding a block expensive in computational terms.
The rule is simple: *the block's hash must start with a certain number of leading zeros*.

Because you cannot predict which hash a given input will produce, miners must brute-force a special field called the **nonce**:

```
nonce = 0
while hash(block_data + nonce) does NOT start with "000":
    nonce += 1
```

The number of required leading zeros is the **difficulty**.  Each extra zero makes mining ~16× harder on average (since each hex digit has 16 possible values).

| Difficulty | Average attempts |
|:----------:|:----------------:|
| 1 | ~16 |
| 2 | ~256 |
| 3 | ~4 096 |
| 4 | ~65 536 |

Bitcoin currently uses a difficulty equivalent to ~20 leading zeros (in binary).

### Task
Mine a new block at difficulty 3 and report how many nonce attempts were needed.

In [ ]:
# ── Part 4: Proof of Work Mining ─────────────────────────────────────────────
def mine_block(
    index: int,
    transactions: list,
    previous_hash: str,
    timestamp: str,
    difficulty: int = 3,
) -> tuple:
    """
    Increment nonce until the block hash starts with `difficulty` leading zeros.
    Returns (mined_block_dict, number_of_attempts).
    """
    block = {
        "index":         index,
        "timestamp":     timestamp,
        "transactions":  transactions,
        "previous_hash": previous_hash,
        "nonce":         0,
    }
    target   = "0" * difficulty
    attempts = 0

    while True:
        attempts += 1
        digest = calculate_block_hash(block)
        if digest.startswith(target):
            block["hash"] = digest
            return block, attempts
        block["nonce"] += 1

mined_block, attempts = mine_block(
    3,
    ["David pays Erin 1"],
    chain[-1]["hash"],
    "2026-10-01T09:03:00Z",
    difficulty=3,
)

print(f"Difficulty : 3 leading zeros  (target prefix = '000')")
print(f"Attempts   : {attempts}")
print(f"Nonce      : {mined_block['nonce']}")
print(f"Hash       : {mined_block['hash']}")

---
## Part 5 – Forking and Longest-Chain Selection

### Background

A **fork** happens when two miners solve the PoW puzzle at almost the same time and broadcast different valid blocks.
The network temporarily has two competing branches:

```
genesis ── block_1 ── block_2 ──┬── block_A  (Branch A: David → Erin)
                                 └── block_B  (Branch B: David → Frank)
```

Both branches are cryptographically valid.  The tie is broken by the **longest-chain rule** (also called Nakamoto consensus):

> *Nodes always adopt the chain with the most blocks (greatest accumulated proof of work).*

Once one branch grows longer, the other is abandoned.  Transactions on the discarded branch are returned to the mempool.

### Task
1. Mine two competing blocks on top of the same parent.
2. Extend Branch A by one more block so it becomes the longest chain.
3. Select the preferred branch using `max(..., key=len)` and display it.

In [ ]:
# ── Part 5: Forking and Longest-Chain Selection ───────────────────────────────
branch_a = deepcopy(chain)
branch_b = deepcopy(chain)

# Two miners find valid blocks on top of the same parent simultaneously
block_a, attempts_a = mine_block(
    3, ["David pays Erin 1"],  branch_a[-1]["hash"],
    "2026-10-01T09:03:00Z", difficulty=3,
)
block_b, attempts_b = mine_block(
    3, ["David pays Frank 1"], branch_b[-1]["hash"],
    "2026-10-01T09:03:01Z", difficulty=3,
)

branch_a.append(block_a)
branch_b.append(block_b)

print("Both forks are valid at this point:")
print(f"  Branch A validity : {validate_chain(branch_a)}")
print(f"  Branch B validity : {validate_chain(branch_b)}")

# Branch A mines one more block → becomes the longest chain
block_a_2, _ = mine_block(
    4, ["Erin pays Grace 1"], branch_a[-1]["hash"],
    "2026-10-01T09:04:00Z", difficulty=3,
)
branch_a.append(block_a_2)

# Longest-chain rule: pick the branch with the most blocks
preferred = max([branch_a, branch_b], key=len)
print(f"\nPreferred chain length : {len(preferred)} blocks  (Branch A wins)")
print()
show_chain(preferred)

---
## Extension – Transaction Validity Rules

### Background

**Cryptographic validation** (hash checks) only guarantees that data has not been *altered after the fact*.
It says nothing about whether the data was *correct to begin with*.

A block can pass all hash checks and still contain transactions that are logically wrong:

| Logically invalid transaction | Why it is wrong |
|-------------------------------|-----------------|
| `Alice pays Bob -10` | Negative amount |
| `Alice pays Alice 5` | Sender == receiver (self-payment) |
| `Alice Bob 10` | Missing keyword `pays` |
| `Alice pays Bob 0` | Zero amount (no value transferred) |

This is why blockchains need **two layers of validation**:
1. **Cryptographic** – did the data change? *(hash-based)*
2. **Business logic** – is the data meaningful? *(rule-based)*

### Task
Write `is_valid_transaction()` and test it against both valid and invalid inputs.  
Then demonstrate that a block with an invalid transaction can still pass cryptographic validation.

In [ ]:
# ── Extension: Transaction Validity Rules ─────────────────────────────────────
def is_valid_transaction(tx: str) -> bool:
    """
    Apply logical rules to a transaction string of the form:
        '<Sender> pays <Receiver> <Amount>'

    Returns True only when:
      - Format is correct (4 tokens, second token is 'pays')
      - Amount is a positive number
      - Sender and receiver are not the same person
    """
    try:
        parts = tx.split(" ")
        if len(parts) != 4 or parts[1] != "pays":
            return False
        sender, _, receiver, amount_str = parts
        amount = float(amount_str)
        if amount <= 0 or sender == receiver:
            return False
        return True
    except Exception:
        return False

# ── Test the validator against a range of inputs ──────────────────────────────
test_cases = [
    ("Alice pays Bob 10",    True,  "valid"),
    ("Alice pays Bob -10",   False, "negative amount"),
    ("Alice pays Alice 5",   False, "self-payment"),
    ("Alice Bob 10",         False, "missing 'pays' keyword"),
    ("Alice pays Bob 0",     False, "zero amount"),
]

print(f"  {'Transaction':<28} {'Expected':<10} {'Got':<10} {'Pass?'}")
print("  " + "-" * 64)
for tx, expected, reason in test_cases:
    result = is_valid_transaction(tx)
    ok = "✓" if result == expected else "✗ FAIL"
    print(f"  {tx:<28} {str(expected):<10} {str(result):<10} {ok}  ({reason})")

# ── Show the two-layer gap ────────────────────────────────────────────────────
print()
invalid_tx = "Alice pays Bob -10"
bad_block  = make_block(3, [invalid_tx], chain[-1]["hash"], "2026-10-01T09:05:00Z")
test_chain = deepcopy(chain)
test_chain.append(bad_block)

print("A block containing a logically invalid transaction:")
print(f"  Cryptographic validation (hash) : {validate_chain(test_chain)}")
print(f"  Logical validation (rules)      : {is_valid_transaction(invalid_tx)}")
print()
print("Conclusion: hash checks alone are insufficient – business rules must also be enforced.")